# Spotify Analytics Dashboard

### Interactive Data Visualization Using Streamlit and Plotly

# Introduction
The purpose of this project is to design and implement an interactive dashboard that demonstrates how multiple visualization techniques can work together to tell a cohesive story from data. The dashboard explores Spotify track characteristics and investigates how musical features such as danceability, energy, acousticness, tempo, and valence relate to song popularity.

Unlike static charts, interactive dashboards allow users to explore the data dynamically by filtering and comparing different groups of songs. This project combines several visualization types into a single application where each visualization responds to user-selected filters, providing a richer and more intuitive analytical experience.

---

# Dataset Description
This project uses the **Spotify Tracks Dataset**, which contains thousands of songs available on Spotify together with their musical characteristics.

The dataset contains the following important variables:

- Track Name
- Artist
- Album
- Track Genre
- Popularity
- Danceability
- Energy
- Loudness
- Speechiness
- Acousticness
- Instrumentalness
- Liveness
- Valence
- Tempo
- Explicit Content Indicator
- Duration

These features make the dataset particularly suitable for exploratory data analysis because they describe different aspects of each song and listener experience.

In [ ]:
import plotly.io as pio
from pathlib import Path
import plotly.express as px

from utils import load_data
import charts as ch

pio.renderers.default = "notebook"

data = load_data("data/spotify_data.csv")
print(data.shape)
data.head()

(113549, 17)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre,duration_min
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,Clean,0.676,0.4610,0.1430,0.0322,0.000001,0.3580,0.715,87.917,acoustic,3.844433
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,Clean,0.420,0.1660,0.0763,0.9240,0.000006,0.1010,0.267,77.489,acoustic,2.493500
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,Clean,0.438,0.3590,0.0557,0.2100,0.000000,0.1170,0.120,76.332,acoustic,3.513767
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,Clean,0.266,0.0596,0.0363,0.9050,0.000071,0.1320,0.143,181.740,acoustic,3.365550
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,Clean,0.618,0.4430,0.0526,0.4690,0.000000,0.0829,0.167,119.949,acoustic,3.314217


---

# Data Cleaning
Before creating the dashboard, several preprocessing steps were performed to improve data quality and usability.

## Remove unnecessary columns
The imported dataset contained an automatically generated index column named:

```
Unnamed: 0
```
Since this column contained no analytical value, it was removed.

```python
df.drop(columns="Unnamed: 0")
```

## Remove duplicate observations
Duplicate songs were removed to prevent repeated records from biasing summary statistics.

```python
df.drop_duplicates()
```

## Handle missing values
Rows containing missing values were removed to ensure that all visualizations and statistical summaries were computed from complete observations.

```python
df.dropna()
```

## Feature Engineering
Song duration was originally stored in milliseconds.

A more interpretable variable called **duration_min** was created.

```python
df["duration_min"] = df["duration_ms"] / 60000
```

The **explicit** variable was also converted into human-readable categories:

- Explicit
- Clean

---

# Visualization Technique
The dashboard combines several visualization techniques that complement one another and allow users to answer different analytical questions.

This notebook keeps the four visualizations used in the current app:

1. Scatter Plot
2. Horizontal Bar Chart
3. Histogram
4. Treemap

## 1. Scatter Plot
**Danceability vs Popularity**

The scatter plot displays the relationship between danceability and popularity while using color to distinguish genres and marker size to represent energy.

This visualization helps answer questions such as:

- Are more danceable songs generally more popular?
- Which genres dominate highly popular songs?
- Are energetic songs clustered together?

To select genres, click or clickout genres from legend on the right

In [52]:
scatter_fig = px.scatter(
    data, 
    x="danceability", 
    y="popularity", 
    color="track_genre", 
    title="Danceability vs Popularity",
    template="plotly_dark" 
)

scatter_fig.update_traces(
    marker=dict(
        size=10,
        line=dict(color="#000000", width=1) 
    )
)

scatter_fig.update_layout(
    xaxis_title="Danceability",
    yaxis_title="Popularity",
    legend_title_text="Genre"
)

scatter_fig.show()


## 2. Horizontal Bar Chart
**Top Artists by Average Popularity**

The bar chart summarizes artists according to their average popularity.

This visualization makes it easy to identify which artists consistently produce highly popular music.

In [45]:
top_artists_fig = ch.top_artists(data)
top_artists_fig.update_traces(marker_color="#d59a7f")
top_artists_fig.show()

## 3. Histogram
**Popularity Distribution**

The histogram displays how popularity scores are distributed across the dataset.

Users can quickly determine whether songs are concentrated at low, medium, or high popularity levels.




In [53]:
distribution_fig = px.histogram(
    data, 
    x="popularity", 
    color="explicit",        
    title="Popularity Distribution by Genre",
    template="plotly_dark",     
    barmode="overlay"          
)

distribution_fig.update_traces(
    opacity=0.75,               
    marker=dict(
        line=dict(color="#000000", width=1) 
    )
)

distribution_fig.update_layout(
    xaxis_title="Popularity",
    yaxis_title="Count",
    legend_title_text="Genre"
)

distribution_fig.show()


## 4. Treemap
**Top Genres by Average Popularity**

The treemap visualizes top genres by average popularity and provides a compact comparison of category performance.

This view complements the artist ranking by showing broader genre context.

In [47]:
genre_fig = ch.genre_popularity(data)
genre_fig.show()

---

# Dashboard Interactivity
One of the major advantages of the dashboard is that all visualizations are linked through common filters.

The sidebar allows users to filter songs by:

- Genre
- Popularity
- Explicit Content

When a filter changes, every visualization updates simultaneously. This coordinated interaction allows users to explore specific subsets of the data without creating separate visualizations.

# Why Streamlit?
Streamlit is an open-source Python framework developed specifically for building interactive data applications. It was selected for simple syntax, fast dashboard development, and strong integration with Pandas and Plotly.

# Why Plotly?
Plotly is an open-source interactive visualization library that supports hover tooltips, zooming, panning, and responsive graphics. This interactivity makes exploratory analysis significantly more effective than static figures.

# Dashboard Framework
The dashboard follows a reactive model: whenever a user changes a filter, Streamlit reruns the app and regenerates all visualizations from the filtered dataset.

# Limitations
- Applications rerun after each interaction, which can slow performance on very large datasets.
- Layout customization is more limited than full JavaScript frameworks.
- Complex client-side interactions are harder to implement.

# Common Issues Encountered
Several implementation challenges were encountered during development.

## Missing Values
Some records contained missing values that prevented visualizations from rendering correctly.

## Duplicate Records
Duplicate tracks affected artist popularity calculations.

## Color Overload
Using too many colors made scatter plots difficult to interpret. Limiting color encoding to genres improved readability.

# Deployment
The dashboard is deployed using **Streamlit Community Cloud**.

1. Push the project to GitHub.
2. Create a Streamlit Community Cloud account.
3. Connect the GitHub repository.
4. Select **app.py** as the application entry point.
5. Deploy the application.

# Conclusion
This project demonstrates how multiple visualization techniques can be integrated into a single interactive dashboard to support exploratory data analysis.

By combining scatter plots, bar charts, histograms, and treemaps within a unified Streamlit application, users can dynamically investigate relationships between musical characteristics and song popularity. Interactive filtering further enhances the analytical experience by allowing all visualizations to update simultaneously, making the dashboard significantly more informative than a collection of static figures.

Overall, this project illustrates the value of interactive visualization in transforming raw data into actionable insights while showcasing modern Python tools for building professional analytical applications.